In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, roc_curve, precision_score, recall_score, f1_score, accuracy_score
from fastcore.basics import *
from fastcore.parallel import *
from os import cpu_count
import pandas as pd

# Exploring Classification the CIC-NIDS-Collection per available attack class with the OneR Model

In [ ]:
df = pd.read_parquet('/kaggle/input/cicidscollection/cic-collection.parquet')
df = df.drop(columns='Label')
df.shape

In [ ]:
df.head()

In [ ]:
feature_info = pd.DataFrame({
    'Feature Name': df.columns,
    'Data Type': df.dtypes
})

# Map data types to categories (e.g., text, number)
feature_info['Type Category'] = feature_info['Data Type'].apply(lambda x: 'Text' if x == 'object' else 'Number')

# Add your additional column to the table

# Display the resulting table
print(feature_info)

# Save to a CSV file
feature_info.to_csv('feature_info.csv', index=False)


# Note: Choose either saving to a file or copying to the clipboard based on your preference.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a larger and clearer figure
plt.figure(figsize=(16, 8))

# Distribution of the target variable
sns.countplot(x='ClassLabel', data=df, palette='viridis')

# Set title and labels
#plt.title('Distribution of the Target Labels', fontsize=16)
plt.xlabel('Target Labels', fontsize=20)
plt.ylabel('Count', fontsize=20)

# Increase the font size of x-axis tick labels
plt.xticks(fontsize=20)  # Adjust the font size according to your preference

plt.savefig('Distribution_target_variables.jpg')
# Show the plot
plt.show()


In [ ]:
df.ClassLabel.value_counts()

**Check for Duplicates**

In [ ]:
# List of labels and their corresponding counts
labels_and_counts = {
    'Benign': 7186189,
    'DDoS': 1234729,
    'DoS': 397344,
    'Botnet': 145968,
    'Bruteforce': 103244,
    'Infiltration': 94857,
    'Webattack': 2995,
    'Portscan': 2255
}

# Check if rows with each label are unique
for label, count in labels_and_counts.items():
    rows_with_label = df[df['ClassLabel'] == label]

    if not rows_with_label.duplicated().any():
        print(f"All {count} rows with label '{label}' are unique.")
    else:
        print(f"There are duplicates in {count} rows with label '{label}'.")
        print(rows_with_label[rows_with_label.duplicated()])
    print("="*50)

In [ ]:
df.drop_duplicates(subset=df.columns[:-1], keep='first')
df.shape

In [ ]:
''# List of labels to keep
labels_to_keep = ['Benign', 'DDoS', 'Bruteforce', 'Botnet']

# Filter the DataFrame to only keep rows with specified labels
df = df[df['ClassLabel'].isin(labels_to_keep)]


In [ ]:
df.head()

In [ ]:
# Create a larger and clearer figure
plt.figure(figsize=(16, 8))

# Distribution of the target variable
sns.countplot(x='ClassLabel', data=df, palette='viridis')

# Set title and labels
#plt.title('Distribution of the Target Labels', fontsize=16)
plt.xlabel('Target Labels', fontsize=20)
plt.ylabel('Count', fontsize=20)

# Increase the font size of x-axis tick labels
plt.xticks(fontsize=20)  # Adjust the font size according to your preference

plt.savefig('Target_distribution_after_drop.png')
# Show the plot
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

# Encode the 'ClassLabel' column to numerical values for correlation
df['ClassLabel'] = df['ClassLabel'].astype('category').cat.codes

label_encoder = LabelEncoder()
df['ClassLabel'] = label_encoder.fit_transform(df['ClassLabel'])

# Calculate the correlation matrix
correlation_matrix = df.corr()

# Plot the heatmap using Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix')
plt.show()


#### 

**Correlation with ClassLabel**

In [ ]:
correlation_matrix['ClassLabel'].drop('ClassLabel').sort_values(ascending=False)
# correlation_matrix

**Keeping highest corealted features**

In [ ]:
positive_correlation_features = [
    'Avg Packet Size', 'Packet Length Mean', 'Bwd Packet Length Std', 'Packet Length Variance',
    'Bwd Packet Length Max', 'Packet Length Max', 'Packet Length Std', 'Fwd Packet Length Mean',
    'Avg Fwd Segment Size', 'Flow Bytes/s', 'Avg Bwd Segment Size', 'Bwd Packet Length Mean',
    'Fwd Packets/s', 'Flow Packets/s', 'Init Fwd Win Bytes', 'Subflow Fwd Bytes',
    'Fwd Packets Length Total', 'Fwd Act Data Packets', 'Total Fwd Packets', 'Subflow Fwd Packets'
    # Add more features as needed
]
df = df[positive_correlation_features + ['ClassLabel']]
df.shape

In [ ]:
df.head()

In [ ]:
df.ClassLabel.value_counts()

In [ ]:
# Distribution of the target variable
sns.countplot(x='ClassLabel', data=df)
plt.title('Distribution of the selected Target Labels')
plt.show()

**undersampling the benign feature beacuse of too many valures**

In [ ]:
# Get the counts of each label
label_counts = df['ClassLabel'].value_counts()

# Set the target count to the count of label 2
target_count = label_counts.min()

# Undersample labels 0 and 1 to the target count
undersampled_df = pd.concat([
    df[df['ClassLabel'] == 0].sample(target_count, replace=False),
    df[df['ClassLabel'] == 1].sample(target_count, replace=False),
    df[df['ClassLabel'] == 2],
    df[df['ClassLabel'] == 3].sample(target_count, replace=False)
], axis=0)

# Shuffle the undersampled DataFrame to mix the labels
undersampled_df = undersampled_df.sample(frac=1).reset_index(drop=True)


In [ ]:
undersampled_df.shape

In [ ]:
label_counts = undersampled_df['ClassLabel'].value_counts()
label_counts

In [ ]:
df=undersampled_df

In [ ]:
# Distribution of the target variable
sns.countplot(x='ClassLabel', data=df)
plt.title('Distribution of the Target Variable')
plt.show()

**ML Algorithms**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

def apply_normalization(X, method='standard'):
    if method == 'standard':
        scaler = StandardScaler()
    elif method == 'minmax':
        scaler = MinMaxScaler()
    elif method == 'robust':
        scaler = RobustScaler()
    else:
        raise ValueError("Invalid normalization method. Choose 'standard', 'minmax', or 'robust'.")
    
    X_normalized = scaler.fit_transform(X)
    return X_normalized

In [ ]:
# Encode the ClassLabel column to numerical values
# label_encoder = LabelEncoder()
# df['ClassLabel'] = label_encoder.fit_transform(df['ClassLabel'])

# Separate features (X) and target variable (y)
X = df.drop('ClassLabel', axis=1)
y = df['ClassLabel']

# Split the data into training, testing, and validation sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.33, random_state=42)



In [ ]:
# Choose the normalization method
normalization_method = 'standard'  # Change this to 'minmax' or 'robust' if needed

# Apply normalization
X_train_normalized = apply_normalization(X_train, method=normalization_method)
X_val_normalized = apply_normalization(X_val, method=normalization_method)

In [ ]:
# Random Forest
rf_model = RandomForestClassifier()
rf_model.fit(X_train_normalized, y_train)


In [ ]:
rf_y_pred = rf_model.predict(X_val_normalized)
rf_y_prob = rf_model.predict_proba(X_val_normalized)
rf_accuracy = accuracy_score(y_val, rf_y_pred)
rf_roc_auc = roc_auc_score(y_val, rf_y_prob, multi_class='ovr')
rf_classification_rep = classification_report(y_val, rf_y_pred)

# Print Random Forest results
print("Random Forest Model")
print(f"Normalization Method: {normalization_method}")
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"ROC AUC: {rf_roc_auc:.4f}")
print("Classification Report:\n", rf_classification_rep)
print("="*50)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Assuming y_val is a categorical variable with multiple classes
y_val_bin = label_binarize(y_val, classes=[0, 1, 2, 3])  # Binarize the labels

# Calculate ROC curve and ROC AUC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(y_val_bin.shape[1]):
    fpr[i], tpr[i], _ = roc_curve(y_val_bin[:, i], rf_y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure(figsize=(8, 6))
for i in range(y_val_bin.shape[1]):
    plt.plot(fpr[i], tpr[i], label=f'Class {i} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)  # Plot diagonal line for reference
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Random Forest Model')
plt.legend(loc='lower right')
plt.show()


In [ ]:
# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train_normalized, y_train)


In [ ]:
lr_y_pred = lr_model.predict(X_val_normalized)
lr_y_prob = lr_model.predict_proba(X_val_normalized)
lr_accuracy = accuracy_score(y_val, lr_y_pred)
lr_roc_auc = roc_auc_score(y_val, lr_y_prob,multi_class='ovr')
lr_classification_rep = classification_report(y_val, lr_y_pred)

# Print Logistic Regression results
print("Logistic Regression Model")
print(f"Normalization Method: {normalization_method}")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"ROC AUC: {lr_roc_auc:.4f}")
print("Classification Report:\n", lr_classification_rep)
print("="*50)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# K-Nearest Neighbors
knn_model = KNeighborsClassifier()
knn_model.fit(X_train_normalized, y_train)



In [ ]:
knn_y_pred = knn_model.predict(X_val_normalized)
knn_accuracy = accuracy_score(y_val, knn_y_pred)

# Since KNN doesn't have predicted probabilities like logistic regression,
# ROC AUC may not be directly applicable for KNN. You can use other metrics like
# precision, recall, and F1-score.

knn_classification_rep = classification_report(y_val, knn_y_pred)

# Print K-Nearest Neighbors results
print("K-Nearest Neighbors Model")
print(f"Normalization Method: {normalization_method}")
print(f"Accuracy: {knn_accuracy:.4f}")
print("Classification Report:\n", knn_classification_rep)
print("="*50)

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Assuming X_train and X_val are your training and validation datasets

# Gaussian Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_normalized, y_train)


In [ ]:
nb_y_pred = nb_model.predict(X_val_normalized)
nb_accuracy = accuracy_score(y_val, nb_y_pred)
nb_classification_rep = classification_report(y_val, nb_y_pred)

# Print Naive Bayes results
print("Gaussian Naive Bayes Model")
print(f"Normalization Method: {normalization_method}")
print(f"Accuracy: {nb_accuracy:.4f}")
print("Classification Report:\n", nb_classification_rep)
print("="*50)


**Deeep Models**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
model = Sequential()
model.add(Dense(128, input_dim=X_train.shape[1], activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(4, activation='softmax'))  # Assuming 4 output classes


In [ ]:
model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train_normalized, y_train, epochs=10, batch_size=32, validation_data=(X_val_normalized, y_val))


In [ ]:
nn_y_prob = model.predict(X_val_normalized)
nn_y_pred = tf.argmax(nn_y_prob, axis=1).numpy()
nn_accuracy = accuracy_score(y_val, nn_y_pred)
nn_classification_rep = classification_report(y_val, nn_y_pred)

# Print Neural Network results
print("Deep Neural Network Model")
print(f"Accuracy: {nn_accuracy:.4f}")
print("Classification Report:\n", nn_classification_rep)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import numpy as np
# Plot ROC Curve
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(4):  # Assuming 4 classes
    fpr[i], tpr[i], _ = roc_curve((y_val == i).astype(int), nn_y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(8, 8))

for i in range(4):
    plt.plot(fpr[i], tpr[i], label=f'Class {i} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.show()

# Plot Confusion Matrix
nn_y_pred = tf.argmax(nn_y_prob, axis=1).numpy()
conf_matrix = confusion_matrix(y_val, nn_y_pred, labels=np.unique(y))
conf_matrix_display = ConfusionMatrixDisplay(conf_matrix, display_labels=np.unique(y))
conf_matrix_display.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.show()


**ML Ensemble**

In [ ]:
import numpy as np
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from keras.wrappers.scikit_learn import KerasClassifier

In [ ]:
class KerasClassifierWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, build_fn, **kwargs):
        self.build_fn = build_fn
        self.kwargs = kwargs
        self.model = None
        self.classes_ = [0, 1]  # Add classes attribute

    def fit(self, X, y):
        self.model = self.build_fn(**self.kwargs)
        self.model.fit(X, y)
        return self

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        # Using the model's predictions as probabilities (for 'soft' voting)
        proba = self.model.predict(X)
        return np.hstack([1 - proba, proba])

    def clone(self):
        return clone(self)


# Define the deep learning model functions
def create_model_1():
    model = Sequential()
    model.add(Dense(units=256, activation='relu', input_dim=X_train_normalized.shape[1]))
    model.add(Dense(units=128, activation='relu'))
    model.add(Dense(units=64, activation='relu'))
    model.add(Dense(units=32, activation='relu'))
    model.add(Dense(units=16, activation='relu'))
    model.add(Dense(units=8, activation='relu'))
    model.add(Dense(units=4, activation='softmax'))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def create_model_2():
    model = Sequential()
    model.add(Dense(units=256, activation='relu', input_dim=X_train_normalized.shape[1]))
    model.add(Dense(units=64, activation='relu'))
    model.add(Dense(units=16, activation='relu'))
    model.add(Dense(units=4, activation='softmax'))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def create_model_3():
    model = Sequential()
    model.add(Dense(units=128, activation='relu', input_dim=X_train_normalized.shape[1]))
    model.add(Dense(units=256, activation='relu'))
    model.add(Dense(units=128, activation='relu'))
    model.add(Dense(units=4, activation='softmax'))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model



# Create Keras classifiers wrapped in a custom wrapper
keras_classifier_1 = KerasClassifierWrapper(build_fn=create_model_1, epochs=10, batch_size=32)
keras_classifier_2 = KerasClassifierWrapper(build_fn=create_model_2, epochs=10, batch_size=32)
keras_classifier_3 = KerasClassifierWrapper(build_fn=create_model_3, epochs=10, batch_size=32)


In [ ]:
ensemble_model = VotingClassifier(estimators=[
    ('keras1', keras_classifier_1),
    ('keras2', keras_classifier_2),
    ('keras3', keras_classifier_3),
], voting='hard')


In [ ]:
ensemble_model.fit(X_train_normalized, y_train)

In [ ]:
# Predictions on the validation set
ensemble_y_pred = ensemble_model.predict(X_val)

# Print ensemble model results
ensemble_accuracy = accuracy_score(y_val, ensemble_y_pred)
ensemble_classification_rep = classification_report(y_val, ensemble_y_pred)

print("Ensemble Model (Voting Classifier)")
print(f"Accuracy: {ensemble_accuracy:.4f}")
print("Classification Report:\n", ensemble_classification_rep)